# Extract unit analysis metrics from Allen Neuropixels cache

**Run this notebook with the `allen-env` kernel** (the one with allensdk installed).

Iterates over every session in `spike_times_v2/`, loads the per-unit analysis
metrics from `EcephysProjectCache`, and saves a single pickle:
`data/unit_analysis_metrics.pkl` — a dict mapping
`session_id → DataFrame` (index = unit_id, columns = whatever the SDK returns).

The vis notebook can then load this file without needing allensdk.

In [ ]:
import pickle
from pathlib import Path

from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache

CACHE_DIR  = Path('/Users/pmccarthy/Documents/experimental_data/allen_visual_neuropixels_longwindow_5ms_bins')
SPIKES_DIR = CACHE_DIR / 'spike_times_v2'
OUT_PATH   = Path('../data/unit_analysis_metrics.pkl')
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

cache = EcephysProjectCache.from_warehouse(manifest=str(CACHE_DIR / 'manifest.json'))

pkl_files = sorted(SPIKES_DIR.glob('*_alllayers_spiketimes.pkl'))
session_ids = [int(p.stem.split('_')[0]) for p in pkl_files]
print(f'{len(session_ids)} sessions to process')

In [ ]:
all_metrics = {}

for i, session_id in enumerate(session_ids, 1):
    print(f'[{i}/{len(session_ids)}] session {session_id} ...', end=' ')
    try:
        # filter_by_validity=False keeps all units, not just those passing QC
        df = cache.get_unit_analysis_metrics_for_session(
            session_id, filter_by_validity=False
        )
        all_metrics[session_id] = df.to_dict(orient='index')
        ns_cols = [c for c in df.columns if 'ns' in c.lower() or 'scene' in c.lower()]
        n_with_pref = sum(
            1 for col in ns_cols if 'pref' in col
            for v in df[col] if v is not None and str(v) != 'nan'
        )
        print(f'OK — {len(df)} units total, NS cols: {ns_cols}, non-NaN pref: {n_with_pref}')
    except Exception as e:
        print(f'ERROR: {e}')

with open(OUT_PATH, 'wb') as f:
    pickle.dump(all_metrics, f)

print(f'\nSaved {len(all_metrics)} sessions → {OUT_PATH.resolve()}')

In [ ]:
# Check coverage: for each session, how many of our VISp pkl units are in the metrics?
import pickle as _pk

ns_col = None
total_visp = 0
total_in_metrics = 0
total_with_pref = 0

for pkl_path in sorted(SPIKES_DIR.glob('*_alllayers_spiketimes.pkl')):
    sid = int(pkl_path.stem.split('_')[0])
    with open(pkl_path, 'rb') as f:
        d = _pk.load(f)
    visp_ids = set(int(u) for u in d['unit_ids'])
    sess_m   = all_metrics.get(sid, {})

    in_metrics  = visp_ids & set(sess_m.keys())
    if not ns_col and in_metrics:
        # find the pref image column name
        sample = sess_m[next(iter(in_metrics))]
        ns_col = next((c for c in sample if 'pref' in c and 'ns' in c.lower()), None)

    with_pref = sum(
        1 for uid in in_metrics
        if ns_col and sess_m[uid].get(ns_col) is not None
        and str(sess_m[uid].get(ns_col)) != 'nan'
    )

    total_visp       += len(visp_ids)
    total_in_metrics += len(in_metrics)
    total_with_pref  += with_pref
    print(f'{sid}: {len(visp_ids)} VISp units, '
          f'{len(in_metrics)} in metrics, {with_pref} with pref_image')

print(f'\nTotals: {total_visp} VISp units | '
      f'{total_in_metrics} in metrics ({100*total_in_metrics/total_visp:.0f}%) | '
      f'{total_with_pref} with pref ({100*total_with_pref/total_visp:.0f}%)')
print(f'Pref image column: {ns_col}')

In [ ]:
# Diagnose pref_image_ns encoding and save condition_id → frame mapping.
#
# pref_image_ns stores stimulus_condition_id, not frame index.
# We build the mapping {condition_id: frame_idx} from the stimulus table and
# save it to data/condition_id_to_frame.json so the vis notebook can use it
# without allensdk.
#
# We use a single session — condition IDs are global Brain Observatory IDs that
# are the same across all sessions (same 118 natural scene images, same IDs).
# Running on multiple sessions and checking they agree would be ideal but is
# expensive (requires re-downloading NWBs).

import json
import pickle as _pk

_sid = session_ids[0]
print(f'Building condition_id → frame mapping from session {_sid}')

_session = cache.get_session_data(_sid)
_ns      = _session.get_stimulus_table('natural_scenes')

# Build mapping: one row per unique condition (each condition = one natural scene image)
_cond_map = (
    _ns[_ns['frame'] >= 0]                          # exclude gray screen (frame == -1)
    .drop_duplicates('stimulus_condition_id')
    [['stimulus_condition_id', 'frame']]
    .set_index('stimulus_condition_id')['frame']
    .astype(int)
    .to_dict()
)

print(f'Mapping has {len(_cond_map)} entries')
print('First 5 (condition_id → frame):',
      dict(list(_cond_map.items())[:5]))

# Verify coverage: every pref_image_ns value in the metrics must be in the map
_all_pref = set(
    int(row['pref_image_ns'])
    for sess in all_metrics.values()
    for row in sess.values()
    if row.get('pref_image_ns') is not None and str(row.get('pref_image_ns')) != 'nan'
)
_missing = _all_pref - set(_cond_map.keys())
print(f'pref_image_ns values not in map: {_missing}')   # should be empty or only gray-screen ID

# Save
_out = Path('../data/condition_id_to_frame.json')
with open(_out, 'w') as _f:
    # JSON keys must be strings
    json.dump({str(k): v for k, v in _cond_map.items()}, _f)
print(f'\nSaved → {_out.resolve()}')

In [ ]:
import numpy as np

# get_natural_scene_template(number) fetches one image at a time (cached locally).
# Frame indices in the stimulus table are 0-117, so we fetch those numbers.
# First call may download; subsequent calls hit the local cache.

_scenes = []
for _n in range(118):
    _img = cache.get_natural_scene_template(_n)
    _scenes.append(_img)
    if _n == 0:
        print(f'Scene 0: type={type(_img)}, ', end='')
        try:
            print(f'shape={_img.shape}, dtype={_img.dtype}')
        except Exception:
            print(_img)

_templates = np.stack(_scenes, axis=0)
print(f'\nStacked templates: {_templates.shape}, dtype={_templates.dtype}')

_tpl_out = Path('../data/natural_scene_templates.npy')
np.save(_tpl_out, _templates)
print(f'Saved → {_tpl_out.resolve()}')